In [ ]:
import pandas as pd
import sqlite3

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
df = pd.read_csv('/content/drive/MyDrive/dados_ist.csv')


In [15]:
# Função Mapper: extrai pares (localidade, 1)
def mapper(linhas):
    pares = []
    for linha in linhas:
        if pd.notna(linha):
            pares.append((linha.strip(), 1))
    return pares

# Função Shuffle: agrupa por localidade
def shuflle(mapped_data):
    agrupado = {}
    for chave, valor in mapped_data:
        if chave not in agrupado:
            agrupado[chave] = []
        agrupado[chave].append(valor)
    return agrupado

# Função Reducer: soma as ocorrências por localidade
def reducer(agrupado):
    return {chave: sum(valores) for chave, valores in agrupado.items()}

# Aplicando MapReduce
mapeado = mapper(df["localidade"])
agrupado = shuflle(mapeado)
resultado = reducer(agrupado)

# Exibindo o resultado
print("Map Reduce: Contagem por localidade")
print(resultado)

Map Reduce: Contagem por localidade
{'Silveira das Flores': 23, 'Peixoto': 487, 'Lopes': 501, 'da Mata de Sales': 1, 'Oliveira do Amparo': 16, 'Brito': 496, 'Viana de Minas': 30, 'Montenegro das Pedras': 25, 'Ferreira de Nascimento': 1, 'da Costa de Barros': 3, 'da Rosa': 522, 'Mendonça das Flores': 15, 'Jesus do Norte': 17, 'Nogueira': 482, 'da Cruz': 497, 'Abreu da Prata': 28, 'da Cruz de Goiás': 17, 'Nunes Verde': 25, 'Duarte do Amparo': 12, 'Vargas': 478, 'Casa Grande': 513, 'Nunes das Pedras': 21, 'Moreira': 516, 'Leão da Prata': 15, 'Martins Alegre': 17, 'Leão': 450, 'Rodrigues': 486, 'Mendes Verde': 19, 'Fonseca de Fogaça': 4, 'Borges da Praia': 27, 'Câmara': 490, 'Camargo': 501, 'Rezende da Serra': 23, 'Caldeira da Serra': 29, 'Cardoso da Serra': 19, 'Souza do Norte': 23, 'Vieira do Campo': 18, 'da Conceição': 486, 'Barros': 514, 'Leão de Siqueira': 1, 'Silveira de Carvalho': 4, 'Duarte das Pedras': 22, 'Macedo': 500, 'Rocha': 472, 'Lopes Verde': 22, 'Almeida da Praia': 16, 'Al

In [16]:
# Hive: SQL pandas
print("Hive: SQL pandas")

# Selecionando colunas relevantes
df_hive = df[["nome", "idade", "renda_media"]]

# Consulta: pessoas com 30 anos ou mais
resultado_hive = df_hive.query("idade >= 30")
print(resultado_hive)

Hive: SQL pandas
                          nome  idade  renda_media
2      Sra. Emilly Vasconcelos     45         8056
3                  Ian da Cruz     59        14504
4                Laís Carvalho     39         5742
5          João Felipe Ribeiro     54        13561
7      Enzo Gabriel Cavalcanti     38        18535
...                        ...    ...          ...
99994     Luiz Fernando Campos     42        10514
99995           Allana Moreira     43         1704
99996       Dra. Ester da Mata     42         9263
99997     Maria Júlia Siqueira     33         1564
99998              Bruno Pinto     39         2913

[71455 rows x 3 columns]


In [17]:
# PIG: Transformações de dados em tempo real
dados_pig = df[["nome", "idade", "renda_media"]].dropna().head(100).values.tolist()

# Filtrar pessoas com 30 anos ou mais
filtrado = [x for x in dados_pig if x[1] >= 30]

# Transformar: calcular índice (exemplo: idade * renda_media / 1000)
transformado  = [(x[0], x[1], x[2], (x[1] * x[2]) / 1000) for x in filtrado]

print("PIG: Transformação de dados em tempo real (idade X renda_media)")
print("Nome | Idade | Renda Média | Índice")

for nome, idade, renda, indice in transformado:
    print(f"{nome} | {idade} | {renda} | {indice:.2f}")


PIG: Transformação de dados em tempo real (idade X renda_media)
Nome | Idade | Renda Média | Índice
Sra. Emilly Vasconcelos | 45 | 8056 | 362.52
Ian da Cruz | 59 | 14504 | 855.74
Laís Carvalho | 39 | 5742 | 223.94
João Felipe Ribeiro | 54 | 13561 | 732.29
Enzo Gabriel Cavalcanti | 38 | 18535 | 704.33
Dr. Thomas Martins | 56 | 18355 | 1027.88
Liam Casa Grande | 44 | 7313 | 321.77
Maria Eduarda Freitas | 35 | 10349 | 362.21
José Pedro Borges | 33 | 3240 | 106.92
Benjamin Fernandes | 56 | 10837 | 606.87
Maria Castro | 59 | 12867 | 759.15
Heloísa da Rosa | 43 | 13530 | 581.79
Bárbara Pimenta | 48 | 1752 | 84.10
João Vargas | 56 | 10883 | 609.45
Breno Marques | 51 | 18921 | 964.97
Alexia Fogaça | 39 | 19280 | 751.92
Thales Dias | 48 | 16563 | 795.02
André Caldeira | 52 | 3707 | 192.76
Luiz Felipe Moraes | 53 | 9108 | 482.72
Matheus Brito | 48 | 9655 | 463.44
Alexandre Campos | 36 | 6375 | 229.50
Thomas Barros | 50 | 1843 | 92.15
Srta. Stephany Porto | 44 | 7320 | 322.08
Lunna Camargo | 30 |

In [21]:
# Simulação do HBase: um banco de dados NoSQL orientado por colunas
# Aqui, usamos um dicionário em Python para simular linhas com chaves únicas e colunas agrupadas por famílias (ex: "info:")

hbase = {
    f"turista:{i}": {  # Cada entrada do dicionário representa uma "linha" do banco, identificada por uma chave do tipo "turista:0", "turista:1", etc.
        "info:idade": float(row["idade"]),           # Coluna na família "info", contendo o valor da idade (convertido para float)
        "info:renda_media": float(row["renda_media"])# Outra coluna na família "info", com a renda média (também em float)
    }
    for i, row in df.iterrows()  # Loop sobre todas as linhas do DataFrame usando iterrows() para acessar índice e linha
}

# Impressão dos dados simulados como se fossem lidos de um banco HBase
print("HBase: Leitura simulada dos dados:")
for chave, colunas in hbase.items():  # Loop em cada chave (linha do banco) e seus dados (colunas)
    print(f"{chave} => idade: {colunas['info:idade']}, renda_media: {colunas['info:renda_media']}")
    # Exibe a chave (ex: "turista:0") e os valores das colunas "idade" e "renda_media" daquela linha



A saída de streaming foi truncada nas últimas 5000 linhas.
turista:95000 => idade: 31.0, renda_media: 14967.0
turista:95001 => idade: 56.0, renda_media: 8340.0
turista:95002 => idade: 51.0, renda_media: 2475.0
turista:95003 => idade: 44.0, renda_media: 10245.0
turista:95004 => idade: 23.0, renda_media: 3274.0
turista:95005 => idade: 59.0, renda_media: 10709.0
turista:95006 => idade: 18.0, renda_media: 16701.0
turista:95007 => idade: 46.0, renda_media: 5466.0
turista:95008 => idade: 28.0, renda_media: 9212.0
turista:95009 => idade: 30.0, renda_media: 8725.0
turista:95010 => idade: 28.0, renda_media: 15558.0
turista:95011 => idade: 55.0, renda_media: 9002.0
turista:95012 => idade: 37.0, renda_media: 10837.0
turista:95013 => idade: 32.0, renda_media: 15547.0
turista:95014 => idade: 49.0, renda_media: 12129.0
turista:95015 => idade: 43.0, renda_media: 7770.0
turista:95016 => idade: 30.0, renda_media: 8133.0
turista:95017 => idade: 34.0, renda_media: 8052.0
turista:95018 => idade: 23.0, ren

In [20]:
# Simulando Sqoop: exportar do DataFrame para SQLite, depois para "HDFS" via CSV
conn = sqlite3.connect('dados.db')

# Usar colunas corretas
df[["id", "localidade", "idade", "renda_media"]].to_sql('dados', conn, if_exists='replace', index=False)

# Leitura dos dados de volta do banco
df_sql = pd.read_sql_query("SELECT * FROM dados", conn)

# Exportar para CSV (simulando HDFS)
df_sql.to_csv('dados_sql.csv', index=False)

# Leitura do CSV (simulando leitura do HDFS)
df_hdfs = pd.read_csv('dados_sql.csv')

print("Dados HDFS:")
print(df_hdfs)

Dados HDFS:
                                         id           localidade  idade  \
0      ff45ef76-ac27-471f-949b-c16517c094f6  Silveira das Flores     26   
1      41949d02-261d-4b5a-bc26-32fc9d61fcf3              Peixoto     21   
2      2ece2ab3-5d38-413b-8ef7-268fdbad9a16                Lopes     45   
3      78f4b78d-525b-4e2c-ab0c-09492ab39a8a     da Mata de Sales     59   
4      a8f6edb9-ab3d-4143-be01-7820d1c9af48   Oliveira do Amparo     39   
...                                     ...                  ...    ...   
99995  68aa3945-802a-4432-8e3d-a3cc3ee43a20              Sampaio     43   
99996  4c3dc41c-c7b4-43ea-8d2f-aed1e1332f0b              Ribeiro     42   
99997  e79396f2-6bba-49b4-b694-ff3336c4debc   da Rocha do Amparo     33   
99998  2131d618-265b-4857-91ea-48a2a318f197  da Cunha das Pedras     39   
99999  d9e14268-5dda-4ce2-957a-f72a8f342c1d               Duarte     25   

       renda_media  
0             8726  
1            11044  
2             8056  
3  